# Ladder Model

We want to test a new matchmaking system for the AI Arena StarCraft 2 bot ladder. To do this, we need a model of the ladder that can simulate match outcomes and durations for any pair of bots.

The model must handle two regimes:

- **Data-rich match-ups:** where we have historical results between two specific bots.
- **Data-poor match-ups:** where we must fall back to ELO-based predictions.

We use Bayesian inference to smoothly blend the two, with the ELO-based prediction as the prior and observed match data as evidence.

## Ratings Updates

The expected score $E_\text{A}$ of a match between bots A and B (from A's perspective) with ratings $R_\text{A}$ and $R_\text{B}$ is given as

$$
E_\text{A} = \frac{1}{1 + 10^{\frac{R_\text{B} - R_\text{A}}{400}}}
$$

and update rule for bot A is

$$
\Delta R_\text{A} = K \cdot ( S_\text{A} - E_\text{A} ),
$$

where $K$ is the adjustment per game (set to 16 on AI Arena) and $S_\text{A}$ is A's actual score of the match (1 for a win, 0.5 for a draw and 0 for a loss).

The expected score $E_\text{A}$ is only a proxy for A's win probability if there is no possibility for a draw.
In the bot games of AI Arena, we frequently encounter draws, and we need to model them.

Additionally, ratings on AI Arena are often **not transitive**, i.e. A beating B almost surely and B beating C a.s. does not imply A beating C a.s. (which the ELO system assumes). For this reason, where we have data available we should use it to inform the outcome of a simulated match.
To get a smooth transition between no-data to data match-ups, we use the ELO system estimate (augmented with a global draw rate, $d$) as the prior and perform Bayesian inference with actual data, where available.

On AI Arena, matches have a **hard time limit of 60 minutes**. Games that reach this limit are always recorded as draws. We model this timeout event first, as it couples match duration and outcome. Concretely, for each simulated match we first determine whether it times out, and then:

- **If timeout:** duration is 60 minutes and the outcome is a draw.
- **If no timeout:** duration and outcome are sampled independently from the models below.

This means that both the outcome model and the duration model are fitted on **non-timeout games only**, and the draw rate $d$ refers to the non-timeout draw rate.

## Match Outcome Model

For the prior we use the 3-dimensional Dirichlet distribution (multivariate beta distribution) which is then updated with the observed number of non-timeout wins ($W$), losses ($L$) and draws ($D$), i.e.

$$
\begin{aligned}
&P_\text{A}^\text{ELO}(\text{win}) = (1 - d) \cdot E_\text{A},\\
&P_\text{A}^\text{ELO}(\text{loss}) = (1 - d) \cdot (1 - E_\text{A}),\\
&P_\text{A}^\text{ELO}(\text{draw}) = d;
\end{aligned}
$$

$$
\boldsymbol{\alpha}_\text{outcome} = \big(n_0 \cdot P_\text{A}^\text{ELO}(\text{win}) + W, \quad n_0 \cdot P_\text{A}^\text{ELO}(\text{draw}) + D, \quad n_0 \cdot P_\text{A}^\text{ELO}(\text{loss}) + L\big),
$$

$$
(p_\text{win},\, p_\text{draw},\, p_\text{loss}) \sim \text{Dir}(\boldsymbol{\alpha}_\text{outcome}),
$$

where $n_0$ determines the strength of the prior (in units of number of matches). In the following I will use $n_0 = 10$, which means that if we have real data for 10 matches, we consider the evidence from the real data as strong as the prior.

## Timeout Model

The timeout probability is modelled with a Beta-Binomial, using the global timeout rate $t$ as prior:

$$
p_\text{timeout} \sim \text{Beta}(n_0 \cdot t + T, \quad n_0 \cdot (1 - t) + N),
$$

where $T$ is the observed number of timeouts and $N$ the number of non-timeouts for the match-up.

## Match Duration Model

For non-timeout games, match durations are positive and right-skewed. We model them with a log-normal distribution truncated at 60 minutes:

$$
\log(\text{duration}) \sim \mathcal{N}(\mu, \sigma^2), \quad \text{duration} < 60.
$$

The truncation ensures that the duration model cannot produce games at the time limit, which are already handled by the timeout model. In practice, since the log-normal is fitted on non-timeout games (all shorter than 60 minutes), the truncation rarely rejects a sample.

The global parameters $\mu_0$ and $\sigma^2$ are fitted from the log-transformed durations of all non-timeout games. For a specific match-up with $n$ observed non-timeout durations with sample mean $\bar{x}$ (in log-space), the posterior mean is

$$
\mu_\text{duration} = \frac{n_0 \cdot \mu_0 + n \cdot \bar{x}}{n_0 + n},
$$

with posterior uncertainty $\sigma_\text{duration} = \sigma / \sqrt{n_0 + n}$. The variance $\sigma^2$ is kept fixed at the global estimate.